# 12 — SRNet curriculum payload feasibility probe (TRAIN only)

This notebook is a **detector-development diagnostic**, not a new RDH experiment.
It was introduced after SRNet-v11 remained near chance on the held-out development split.

Frozen publication quantities remain unchanged:

\[
\alpha=0.25,\qquad R^*_{net}=0.009\ \mathrm{bpp}.
\]

The notebook probes stronger **TRAIN-only curriculum payloads** using the same frozen reversible codec and deterministic random allocation. It does **not** read TEST image pixels, train SRNet, or score TEST.


In [ ]:
from pathlib import Path
import hashlib, json, time, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdhlab.io import read_gray
from rdhlab.blockcodec import analyze_blocks
from rdhlab.pipeline import run_frozen_image_precomputed
from rdhlab.freeze_protocol import sha256_file, stable_id_hash
from rdhlab.srnet_payload_probe_v12 import (
    PayloadSelectionRule,
    curriculum_path,
    deterministic_probe_indices,
    deterministic_random_order,
    select_strong_payload,
    summarize_probe,
)

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
seed=int(config['project']['seed'])
manifest=pd.read_csv(config['dataset']['prepared_manifest'])
train=manifest[manifest.split=='train'].reset_index(drop=True)
test=manifest[manifest.split=='test'].reset_index(drop=True)
assert len(train)==6000 and len(test)==2000
assert train.source_id.astype(str).is_unique
assert test.source_id.astype(str).is_unique
assert set(train.source_id.astype(str)).isdisjoint(set(test.source_id.astype(str)))

allocator_path=Path('/workspace/config/frozen_allocator.json')
allocator=json.loads(allocator_path.read_text())
alpha=float(allocator['alpha'])
bs=int(allocator.get('block_size',config['dataset']['block_size']))
assert np.isclose(alpha,0.25)
assert np.isclose(float(allocator['teacher_payload_bpp']),0.009)

complete_path=Path('/workspace/results/frozen_test_final/test_run_complete.json')
complete=json.loads(complete_path.read_text())
assert stable_id_hash(test.source_id.astype(str).tolist())==complete['test_source_ids_sha256']

OUT=Path('/workspace/results/srnet_payload_probe_v12')
OUT.mkdir(parents=True,exist_ok=True)

print('Train/test manifest:',len(train),len(test))
print('Frozen alpha:',alpha)
print('Frozen primary payload:',allocator['teacher_payload_bpp'])
print('Block size:',bs)
print('NOTE: TEST source IDs are audited, but TEST image pixels are never read by this notebook.')


In [ ]:
# Pre-specified before any probe result is observed.
FIT_POOL_N=5250
PROBE_N=500
PROBE_PAYLOADS=[0.012,0.015,0.020,0.030,0.050]
REFERENCE_BPP=0.012
TARGET_BPP=0.009
MIN_FEASIBLE_FRACTION=0.90
PROBE_SEED=seed+12000

fit_pool=train.iloc[:FIT_POOL_N].reset_index(drop=True)
dev=train.iloc[FIT_POOL_N:].reset_index(drop=True)
assert len(fit_pool)==5250 and len(dev)==750
assert set(fit_pool.source_id.astype(str)).isdisjoint(set(dev.source_id.astype(str)))
assert set(fit_pool.source_id.astype(str)).isdisjoint(set(test.source_id.astype(str)))

idx=deterministic_probe_indices(len(fit_pool),PROBE_N,PROBE_SEED)
probe=fit_pool.iloc[idx].reset_index(drop=True)

protocol={
    'analysis_status':'TRAIN_ONLY_SRNET_CURRICULUM_PAYLOAD_FEASIBILITY_PROBE_V12',
    'target_journal':'Signal Processing',
    'purpose':'select a stronger detector-training curriculum payload before another SRNet CPU run',
    'probe_payloads_net_bpp':PROBE_PAYLOADS,
    'reference_payload_bpp':REFERENCE_BPP,
    'frozen_primary_payload_bpp':TARGET_BPP,
    'probe_n':PROBE_N,
    'fit_pool_n':FIT_POOL_N,
    'probe_seed':PROBE_SEED,
    'probe_source_ids_sha256':stable_id_hash(probe.source_id.astype(str).tolist()),
    'selection_rule':'largest pre-specified payload > 0.012 with feasible_fraction >= 0.90 and exact recovery for every feasible case',
    'min_feasible_fraction':MIN_FEASIBLE_FRACTION,
    'allocation_strategy':'deterministic random allocation for detector development only',
    'allocator_alpha_frozen':alpha,
    'allocator_sha256':sha256_file(allocator_path),
    'test_source_ids_sha256':complete['test_source_ids_sha256'],
    'development_partition_used':False,
    'test_pixels_read':False,
    'test_split_scored':False,
    'srnet_trained':False,
    'no_allocator_or_publication_payload_retuning_permitted':True,
}
protocol_path=OUT/'srnet_v12_payload_probe_protocol.json'
protocol_path.write_text(json.dumps(protocol,indent=2),encoding='utf-8')
protocol_sha=sha256_file(protocol_path)
print('Protocol SHA256:',protocol_sha)
print('Probe IDs:',len(probe),'from TRAIN fitting pool only')
print('Payloads:',PROBE_PAYLOADS)


## Feasibility sweep

For every `(source image, payload)` pair we run the frozen reversible codec with a deterministic random block order. The probe records feasibility and exact recovery before considering any payload eligible for SRNet curriculum training.


In [ ]:
def get_float(d,key,default=np.nan):
    try:
        v=d.get(key,default)
        return float(v) if v is not None else float(default)
    except Exception:
        return float(default)

def get_int(d,key,default=np.nan):
    try:
        v=d.get(key,default)
        return int(v) if v is not None and np.isfinite(float(v)) else default
    except Exception:
        return default

rows=[]
t0_all=time.perf_counter()
for p_idx,payload in enumerate(PROBE_PAYLOADS,1):
    t0=time.perf_counter()
    feasible_count=0
    for j,row in probe.iterrows():
        sid=str(row.source_id)
        x=read_gray(row.path)
        plans=analyze_blocks(x,bs)
        block_ids=[plan.block_id for plan in plans]
        order=deterministic_random_order(block_ids,seed=PROBE_SEED,source_id=sid,payload_bpp=payload)

        case_t0=time.perf_counter()
        rr=run_frozen_image_precomputed(
            x,sid,float(payload),'random',{'random':order},[],bs,seed,False,None,plans=plans
        )
        elapsed_ms=(time.perf_counter()-case_t0)*1000.0
        feasible=bool(rr.get('feasible',False))
        if feasible:
            feasible_count+=1
            exact_image=bool(rr.get('exact_image',False))
            exact_message=bool(rr.get('exact_message',False))
            if not (exact_image and exact_message and float(rr.get('ber',1.0))==0.0):
                raise RuntimeError(f'Exact-recovery invariant failed: source={sid}, payload={payload}')
        else:
            exact_image=False
            exact_message=False

        sideinfo=get_float(rr,'sideinfo_bits')
        gross=get_float(rr,'gross_payload_bits')
        side_fraction=(sideinfo/gross) if np.isfinite(sideinfo) and np.isfinite(gross) and gross>0 else np.nan
        net_bits=get_float(rr,'net_payload_bits')
        actual_net_bpp=(net_bits/(x.shape[0]*x.shape[1])) if np.isfinite(net_bits) else np.nan

        rows.append({
            'source_id':sid,
            'target_net_bpp':float(payload),
            'feasible':feasible,
            'exact_image':exact_image,
            'exact_message':exact_message,
            'ber':get_float(rr,'ber'),
            'actual_net_bpp':actual_net_bpp,
            'psnr':get_float(rr,'psnr'),
            'ssim':get_float(rr,'ssim'),
            'gross_payload_bits':gross,
            'net_payload_bits':net_bits,
            'sideinfo_bits':sideinfo,
            'sideinfo_fraction_gross':side_fraction,
            'used_blocks':get_float(rr,'used_blocks'),
            'changed_pixels':get_float(rr,'changed_pixels'),
            'elapsed_ms':elapsed_ms,
        })
        if (j+1)%100==0 or j+1==len(probe):
            print(f'payload={payload:.3f} | {j+1:03d}/{len(probe)} | feasible={feasible_count}',flush=True)

    print(
        f'PAYLOAD COMPLETE {payload:.3f}: feasible={feasible_count}/{len(probe)} '
        f'({feasible_count/len(probe):.3f}), time={(time.perf_counter()-t0)/60:.2f} min',
        flush=True,
    )

per_case=pd.DataFrame(rows)
per_case.to_csv(OUT/'srnet_v12_payload_probe_per_case.csv',index=False)
print('Total probe time [min]:',(time.perf_counter()-t0_all)/60)


In [ ]:
summary=summarize_probe(per_case)
summary.to_csv(OUT/'srnet_v12_payload_probe_summary.csv',index=False)
display(summary)

rule=PayloadSelectionRule(
    reference_bpp=REFERENCE_BPP,
    min_feasible_fraction=MIN_FEASIBLE_FRACTION,
    require_exact_recovery=True,
)
selection=select_strong_payload(summary,rule)
selection.update({
    'protocol_sha256':protocol_sha,
    'probe_source_ids_sha256':protocol['probe_source_ids_sha256'],
    'test_split_scored':False,
    'srnet_trained':False,
    'allocator_retuned':False,
})
(OUT/'srnet_v12_payload_selection.json').write_text(json.dumps(selection,indent=2),encoding='utf-8')

path=curriculum_path(selection.get('selected_payload_bpp'),PROBE_PAYLOADS,target_bpp=TARGET_BPP,reference_bpp=REFERENCE_BPP)
plan={
    'status':'CURRICULUM_READY' if path else 'NO_CURRICULUM_READY',
    'selected_payload_bpp':selection.get('selected_payload_bpp'),
    'recommended_descending_payload_path':path,
    'target_payload_bpp':TARGET_BPP,
    'reference_payload_bpp':REFERENCE_BPP,
    'note':'TRAIN-only detector-development plan. Patch 13 must still use a held-out development sanity gate before any TEST scoring.',
    'protocol_sha256':protocol_sha,
}
(OUT/'srnet_v12_curriculum_plan.json').write_text(json.dumps(plan,indent=2),encoding='utf-8')

print(json.dumps(selection,indent=2))
print(json.dumps(plan,indent=2))


In [ ]:
# Publication-independent diagnostic figures.
fig,ax=plt.subplots(figsize=(6.2,4.2))
ax.plot(summary.target_net_bpp,summary.feasible_fraction,marker='o')
ax.axhline(MIN_FEASIBLE_FRACTION,linestyle='--',linewidth=1)
ax.set_xlabel('Target net payload [bpp]')
ax.set_ylabel('Feasible fraction on TRAIN probe')
ax.set_ylim(0,1.03)
ax.set_title('SRNet curriculum payload feasibility probe')
fig.tight_layout()
fig.savefig(OUT/'srnet_v12_feasibility.png',dpi=300)
plt.show()

if 'psnr_mean' in summary.columns:
    fig,ax=plt.subplots(figsize=(6.2,4.2))
    ax.plot(summary.target_net_bpp,summary.psnr_mean,marker='o')
    ax.set_xlabel('Target net payload [bpp]')
    ax.set_ylabel('Mean PSNR [dB], feasible TRAIN cases')
    ax.set_title('Distortion at candidate curriculum payloads')
    fig.tight_layout()
    fig.savefig(OUT/'srnet_v12_psnr.png',dpi=300)
    plt.show()


## Stop condition

This notebook intentionally ends here. It **must not train SRNet or score TEST**.

Send the following two files/results for the next decision:

- `srnet_v12_payload_probe_summary.csv`
- `srnet_v12_payload_selection.json`

Only after reviewing these TRAIN-only results should a new SRNet training patch be created.
